In [16]:
import os
import re
import json
import pandas as pd
import time
import pycountry

In [17]:
from google import genai
from dotenv import load_dotenv
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

In [18]:
load_dotenv()
def get_gemini_response(text_data:str):
    try:
        # Create a client
        client = genai.Client(api_key=os.environ["API_KEY"])

        # Send a test prompt
        response = client.models.generate_content(
            model="gemini-2.0-flash",
            contents= text_data
        )

        return response.text

    except Exception as e:
        return f"Error Occured: {str(e)}"

In [19]:
STRUCTURED_FIELDS = [
    "Town Name",
    "Postal Code",
    "Remaining Address",
    "Country Code (2 characters)"
]

In [20]:
# --- Configuration ---

MAX_WORKERS = 30     # Number of parallel threads for processing
MAX_RETRIES = 3      # Minimum retry attempts for LLM calls
SAMPLE_LIMIT = None  # If None process all; else process first N addresses

print(f"Using {MAX_WORKERS} parallel workers. Retries: {MAX_RETRIES}. Sample limit: {SAMPLE_LIMIT}")

Using 30 parallel workers. Retries: 3. Sample limit: None


In [21]:
# Retry helpers
def send_request_with_retry(prompt, validate_fn=None, max_attempts=MAX_RETRIES, delay_base=0.5):
    last_resp = ""
    for attempt in range(1, max_attempts + 1):
        try:
            resp = get_gemini_response(prompt)
        except Exception:
            resp = ""

        last_resp = resp
        ok = False

        if validate_fn:
            try:
                ok = validate_fn(resp)
            except Exception:
                ok = False
        else:
            ok = bool(resp.strip())

        if ok:
            return resp

        time.sleep(delay_base * attempt)

    return last_resp

In [22]:
# Reference data loader (cached)
_reference_df = None

def load_country_reference():
    global _reference_df
    if _reference_df is None:
        base_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
        csv_path = os.path.join(base_dir, 'data', 'input', 'address_formats.csv')
        _reference_df = pd.read_csv(csv_path)
        _reference_df['country_norm'] = _reference_df['country'].str.strip().str.upper()
    return _reference_df

In [23]:
# Country identification (fixed strip line)
def identify_country(address: str, verbose: bool = False, get_prompt: bool = False, llm_response: str = None) -> str:
    df = load_country_reference()
    prompt = f"""You are a highly accurate geolocation API. Your sole function is to identify the country name.
You are given an unstructured address:\n\n{address}\n\nReturn ONLY the country name that best matches the address.
\nWhen resolving conflicting information, prioritize a well-known city name. However, this rule should be ignored
if another part of the address (like a state, province, or a uniquely formatted postal code) provides a definitive
and unambiguous signal for a different country.\nDo not output explanations, JSON, codes, or multiple countries. \nAnswer:"""

    if get_prompt:
        return prompt

    if llm_response is None:
        return ""

    raw = llm_response
    if verbose:
        print("--- COUNTRY ID LLM RESPONSE ---")
        print(raw)
        print("-------------------------------\n")

    cleaned = re.sub(r"^```.*?```", "", raw, flags=re.DOTALL)

    # Handle cases where there might be no lines
    lines = cleaned.splitlines()
    if not lines:
        return ""
    cleaned = lines[0].strip()

    cleaned = cleaned.strip(" '\n\t")

    # Token matching
    df_norm = df['country_norm'].values
    for token in re.split(r"[,.\s/]+", cleaned):
        try:
            token = token.strip()
            if token.upper() in df_norm:
                return df.loc[df['country_norm'] == token.upper(), 'country'].iloc[0]
        except LookupError:
            pass

    for c in df['country']:
        if c.lower() in cleaned.lower():
            return c

    return ""

In [24]:
# Prompt builder (no hallucination, literal substrings only)
def build_structuring_prompt(address: str, country: str, ref_row: pd.Series) -> str:
    ref_info = ref_row.get("reference_information", "") or ""
    examples = ref_row.get("examples", "") or ""
    add_info = ref_row.get("additional_information", "") or ""

    fields_to_extract = ["Town Name", "Postal Code"]

    prompt = f"""
You are given an unstructured address and optional country reference material.
Only extract fields as exact substrings from the ORIGINAL address string (verbatim).
Do NOT invent, expand, translate, normalize, or add information not present in the original address text.

Original address:
{address}

Country: {country}

Reference (may be empty, do not introduce new text from it):
{ref_info}

Examples (IGNORE for adding new tokens, only guidance):
{examples}

Additional notes (IGNORE for adding new tokens):
{add_info}

Return STRICT JSON with keys:
{', '.join(fields_to_extract)}

Rules:
- Each field must be an exact substring (continuous) of the original address or empty string.
- If absent, use "".
- Preserve original substring casing exactly.
- Do not output explanations or markdown.

Example (format only):
{{
    "Town Name": "London",
    "Postal Code": "NW1 6XE"
}}

Now output only the JSON:
"""
    return prompt


In [25]:
# ISO mapping & helper
_fallback_iso = {
    "United States": "US", "United Kingdom": "GB", "Russia": "RU", "Germany": "DE",
    "France": "FR", "Italy": "IT", "Spain": "ES", "Netherlands": "NL", "Belgium": "BE",
    "Austria": "AT", "Switzerland": "CH", "Poland": "PL", "Portugal": "PT", "Romania": "RO",
    "Ukraine": "UA", "Saudi Arabia": "SA", "China": "CN", "Japan": "JP", "Vietnam": "VN",
    "India": "IN", "Brazil": "BR", "Argentina": "AR", "Canada": "CA", "Mexico": "MX",
    "Chile": "CL", "Singapore": "SG", "South Korea": "KR", "Norway": "NO", "Sweden": "SE",
    "Denmark": "DK", "Finland": "FI", "Ireland": "IE", "Greece": "GR", "Turkey": "TR",
    "Türkiye": "TR", "Belarus": "BY", "Bangladesh": "BD", "Hong Kong": "HK", "Macao": "MO",
    "Malaysia": "MY", "New Zealand": "NZ", "Serbia": "RS", "Slovakia": "SK",
    "Slovenia": "SI", "Iceland": "IS", "Israel": "IL"
}

def country_to_iso2(country: str) -> str:
    if pycountry:
        try:
            return pycountry.countries.lookup(country).alpha_2
        except LookupError:
            pass
    return _fallback_iso.get(country, "")

In [26]:
# Robust parse + normalization
def parse_structured_response(text: str) -> dict:
    raw = text.strip()
    raw = re.sub(r"```(?:json)?```", "", raw)
    raw = raw.strip("`\n")

    candidate = raw

    if '{' not in candidate:
        # Try last line
        lines = [l for l in raw.splitlines() if '{' in l]
        if lines:
            candidate = lines[-1]

    if '{' in candidate:
        m = re.search(r"\{.*\}", candidate, flags=re.DOTALL)
        if m:
            candidate = m.group(0)

    try:
        data = json.loads(candidate)
    except Exception:
        data = {}

    result = {}
    fields_to_parse = ["Town Name", "Postal Code"]

    for f in fields_to_parse:
        v = data.get(f, "")
        if v is None:
            v = ""
        result[f] = str(v)

    return result


In [27]:
# Main processing pipeline
def process_address(address_index: int, address: str, verbose: bool = False) -> dict:
    if verbose:
        print(f"=== PROCESSING ADDRESS: {address} ===")

    # --- 1. Country Identification with retry ---
    country_prompt = identify_country(address, verbose=verbose, get_prompt=True)
    country_response = send_request_with_retry(
        country_prompt,
        validate_fn=lambda r: bool(identify_country(address, llm_response=r))
    )

    country = identify_country(address, llm_response=country_response)

    if not country and verbose:
        print("Country identification failed after retries.")

    if not country:
        # Still proceed with empty country (will produce empty ISO)
        # but do NOT abort; structuring may still find town/postal
        pass

    # --- 2. Structuring with retry ---
    iso_code = country_to_iso2(country) if country else ""

    ref_df = load_country_reference()
    ref_row = ref_df.loc[ref_df["country"] == country].iloc[0] \
        if country in ref_df["country"].values else ref_df.iloc[0]

    structuring_prompt = build_structuring_prompt(address, country, ref_row)

    structuring_response = send_request_with_retry(
        structuring_prompt,
        validate_fn=lambda r: any(
            parse_structured_response(r).get(f)
            for f in ["Town Name", "Postal Code"]
        )
    )

    structured = parse_structured_response(structuring_response)

    # --- 3. Hybrid field calculation ---
    remaining_address = address or ""

    town = structured.get("Town Name", "")
    postal_code = structured.get("Postal Code", "")

    if town and town in remaining_address:
        remaining_address = remaining_address.replace(town, "")

    if postal_code and postal_code in remaining_address:
        remaining_address = remaining_address.replace(postal_code, "")

    remaining_address = re.sub(r"\s+", " ", remaining_address).strip().strip(",;")[:140]

    final_result = {
        "Town Name": town,
        "Postal Code": postal_code,
        "Remaining Address": remaining_address,
        "Country Code (2 characters)": iso_code,
    }

    if not town and not postal_code and verbose:
        print(f"Structuring empty after retries for address: {address}")

    return final_result


In [28]:
# Batch processor (modified): returns dict {original_index: result} without padding


def process_addresses_batch(indexed_addresses: list, verbose: bool = False,
                            max_workers: int = MAX_WORKERS) -> dict:
    results = {}
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_index = {
            executor.submit(process_address, orig_idx, addr, verbose): orig_idx
            for orig_idx, addr in indexed_addresses
        }
        for fut in tqdm(as_completed(future_to_index),
                        total=len(indexed_addresses),
                        desc="Processing addresses"):
            idx = future_to_index[fut]
            try:
                results[idx] = fut.result()
            except Exception as exc:
                if verbose:
                    print(f"Address index {idx} exception: {exc}")
                # Failure still yields an empty record for this processed row (keeps 1:1 for processed set only)
                results[idx] = {f: "" for f in STRUCTURED_FIELDS}

    # Ensure every submitted index has a result
    missing = {i for i, _ in indexed_addresses} - set(results.keys())
    if missing:
        raise RuntimeError(f"Missing results for indices: {missing}")
    return results


In [29]:
# Helper: all DataFrame values -> strings, NaN/None -> ""
def enforce_string_df(df: pd.DataFrame) -> pd.DataFrame:
    return df.map(lambda x: "" if (x is None or (isinstance(x, float) and pd.isna(x))) else str(x))

In [30]:
# Process selected subset (no padding to full input length)
print("Loading input data...")
input_excel_path = os.path.join(os.getcwd(), 'data', 'input', 'reference_address_cropped_Unstructured_col.xlsx')
df_input = pd.read_excel(input_excel_path, dtype=str).fillna("")
df_input = enforce_string_df(df_input)
df_input['row_id'] = df_input.index.astype(str)

# Determine indices to process (sample-aware). Only these will appear in results.
if SAMPLE_LIMIT is not None:
    to_process_indices = df_input.index[:SAMPLE_LIMIT]
    print(f"Sample limit active: processing first {SAMPLE_LIMIT} rows out of {len(df_input)} total.")
else:
    to_process_indices = df_input.index

indexed_addresses = [
    (i, df_input.loc[i, 'Unstructured Address'] if pd.notna(df_input.loc[i, 'Unstructured Address']) else "")
    for i in to_process_indices
]

print(f"Found {len(indexed_addresses)} addresses to process.")

print("Starting batch processing... (this may take a while)")
start_time = time.time()
results_dict = process_addresses_batch(indexed_addresses, verbose=False, max_workers=MAX_WORKERS)
end_time = time.time()
print("Batch processing finished.")

processing_time = end_time - start_time
processed_address_count = len(results_dict)
print(f"Total processing time: {processing_time:.2f} seconds")

if processed_address_count:
    print(f"Average time per processed address: {processing_time / processed_address_count:.2f} seconds")

# Build results frame ONLY for processed rows (index preserved)
df_results = pd.DataFrame.from_dict(results_dict, orient='index').sort_index()
df_results.index.name = 'row_id'
df_results = enforce_string_df(df_results)

output_dir = os.path.join(os.getcwd(), 'data', 'output')
os.makedirs(output_dir, exist_ok=True)
output_csv_path = os.path.join(output_dir, 'structured_advanced_pipeline_subset.csv')
df_results.to_csv(output_csv_path)

print(f"Results (processed subset) saved to {output_csv_path}")

# Lightweight success proxy: rows having at least one non-empty extracted field
successful_rows = (
    df_results[["Town Name", "Postal Code"]]
    .apply(lambda r: any(v.strip() for v in r), axis=1)
    .sum()
)
success_rate = successful_rows / processed_address_count if processed_address_count else 0

print(f"Success rate (processed subset): {success_rate:.2%}")
cols = [
    "Town Name",
    "Postal Code",
    "Country Code (2 characters)"
]

df_results[cols] = df_results[cols].apply(lambda c: c.str.lower())


Loading input data...
Found 11 addresses to process.
Starting batch processing... (this may take a while)


Processing addresses: 100%|██████████| 11/11 [00:07<00:00,  1.41it/s]

Batch processing finished.
Total processing time: 7.81 seconds
Average time per processed address: 0.71 seconds
Results (processed subset) saved to c:\Users\varun\varun_personal\ESMT_Test\data\output\structured_advanced_pipeline_subset.csv
Success rate (processed subset): 100.00%


In [31]:
# Comparison & mismatch evaluation (processed subset only)
def assert_processed_alignment(processed_indices, df_results):
    missing = set(processed_indices) - set(df_results.index)
    if missing:
        raise ValueError(f"Missing predictions for processed indices: {missing}")
    return True


print("Loading ground truth data for comparison (subset)...")
ground_truth_path = os.path.join(os.getcwd(), 'data', 'input', 'reference_address_cropped_Unstructured_col.xlsx')
df_ground_truth = pd.read_excel(ground_truth_path, dtype=str).fillna("")
df_ground_truth = enforce_string_df(df_ground_truth)

processed_indices = sorted(results_dict.keys())
assert_processed_alignment(processed_indices, df_results)

df_processed_input = df_input.loc[processed_indices].copy()
df_processed_input = enforce_string_df(df_processed_input)

df_processed_gt = df_ground_truth.loc[df_ground_truth.index.intersection(processed_indices)].copy()
df_processed_gt = enforce_string_df(df_processed_gt)


# Build ground truth maps for subset only
gt_maps = {}
for col in ['Town Name', 'Postal Code', 'Country Code (2 characters)']:
    if col in df_processed_gt.columns:
        gt_maps[col] = df_processed_gt[col].to_dict()
    else:
        gt_maps[col] = {}


for col in ['Town Name', 'Postal Code', 'Country Code (2 characters)']:
    df_processed_input[f"{col}_ground_truth"] = df_processed_input.index.map(
        lambda i: gt_maps[col].get(i, "")
    )


# Attach predictions
df_processed_input['Predicted Town Name'] = df_results['Town Name']
df_processed_input['Predicted Postal Code'] = df_results['Postal Code']
df_processed_input['Predicted Country Code'] = df_results['Country Code (2 characters)']


# Build comparison frame (processed subset only)
df_comparison = df_processed_input[
    [
        'Unstructured Address',
        'Town Name_ground_truth', 'Postal Code_ground_truth', 'Country Code (2 characters)_ground_truth',
        'Predicted Town Name', 'Predicted Postal Code', 'Predicted Country Code'
    ]
].copy()

df_comparison = enforce_string_df(df_comparison)


mismatches = []

comparison_fields = {
    'Town Name': ('Town Name_ground_truth', 'Predicted Town Name'),
    'Postal Code': ('Postal Code_ground_truth', 'Predicted Postal Code'),
    'Country Code': ('Country Code (2 characters)_ground_truth', 'Predicted Country Code')
}

for _, row in df_comparison.iterrows():
    for field_name, (truth_col, pred_col) in comparison_fields.items():
        truth_val = str(row[truth_col]).strip() if row[truth_col] else ""
        pred_val = str(row[pred_col]).strip() if row[pred_col] else ""

        if not truth_val:
            # Skip if ground truth missing for that field (row retained globally)
            continue

        if truth_val != pred_val:
            mismatches.append({
                'Unstructured Address': row['Unstructured Address'],
                'Mismatched Field': field_name,
                'Original Value': truth_val,
                'Predicted Value': pred_val
            })


print(f"Processed rows: {len(df_comparison)} (subset only, no padding).")

if mismatches:
    df_mismatches = pd.DataFrame(mismatches)
    print("\nFound mismatches (first 50):")
    #display(df_mismatches.head(50))
else:
    print("\nNo mismatches found in processed subset.")


Loading ground truth data for comparison (subset)...
Processed rows: 11 (subset only, no padding).

Found mismatches (first 50):


In [32]:
display(df_processed_input)

,record_id,Unstructured Address,house,house_number,road,po_box,unit,level,staircase,suburb,...,near,Country Code (2 characters),source,row_id,Town Name_ground_truth,Postal Code_ground_truth,Country Code (2 characters)_ground_truth,Predicted Town Name,Predicted Postal Code,Predicted Country Code
0,1,The Book Club 100-106 Leonard St Shoreditch Lo...,the book club,100-106,leonard st,,,,,shoreditch,...,,gb,libpostal,0,london,ec2a 4rh,gb,london,ec2a 4rh,gb
1,2,"Royal Opera House, Bow St, Covent Garden, Lond...",royal opera house,,bow st,,,,,covent garden,...,,gb,libpostal,1,london,wc2e 9dd,gb,london,wc2e 9dd,gb
2,3,"Columbus, OH",,,,,,,,,...,,us,libpostal,2,columbus,,us,columbus,,us
3,4,San Francisco CA,,,,,,,,,...,,us,libpostal,3,san francisco,,us,san francisco,,us
4,5,IXe arrondissement Paris,,,,,,,,,...,,fr,libpostal,4,paris,,fr,paris,,fr
5,6,9e arrondissement Paris,,,,,,,,,...,,fr,libpostal,5,paris,,fr,paris,,fr
6,7,Eschenbrau Braurei Triftstrasse 67 13353 Berli...,eschenbrau braurei,67,triftstrasse,,,,,,...,,de,libpostal,6,berlin,13353,de,berlin,13353,
7,8,"Museo del Prado C. de Ruiz de Alarcón, 23 2801...",museo del prado,23,c. de ruiz de alarcón,,,,,,...,,es,libpostal,7,madrid,28014,es,madrid,28014,es
8,9,"Paseo de la Castellana, 185 - 5º, 28046 Madrid...",,185,paseo de la castellana,,,5º,,,...,,es,libpostal,8,madrid,28046,es,madrid,28046,es
9,10,"506/465, Thiruthangal village, Sivakasi, Virud...",,506/465,thiruthangal village,,,,,,...,,in,Libpostal-bug,9,sivakasi,626130,in,sivakasi,626130,in


In [33]:
# Per-field accuracy metrics (processed subset only)
field_mapping = {
    'Town Name': ('Town Name_ground_truth', 'Predicted Town Name'),
    'Postal Code': ('Postal Code_ground_truth', 'Predicted Postal Code'),
    'Country Code (2 characters)': ('Country Code (2 characters)_ground_truth', 'Predicted Country Code')
}

metrics = []
for display_name, (truth_col, pred_col) in field_mapping.items():
    mask = df_comparison[truth_col].astype(str).str.strip() != ""
    total = int(mask.sum())

    if total == 0:
        matches = 0
        acc = 0.0
    else:
        col_truth = df_comparison.loc[mask, truth_col].astype(str).str.strip()
        col_pred = df_comparison.loc[mask, pred_col].astype(str).str.strip()
        matches = int((col_truth == col_pred).sum())
        acc = matches / total

    metrics.append({
        'Field': display_name,
        'Matches': matches,
        'Total': total,
        'Accuracy': round(acc, 4)
    })

df_field_accuracy = pd.DataFrame(metrics)
df_field_accuracy = enforce_string_df(df_field_accuracy)

print("Per-field accuracy (processed subset):")
display(df_field_accuracy)

overall = (
    df_field_accuracy['Matches'].astype(str).str.strip().astype(int).sum()
    /
    df_field_accuracy['Total'].astype(str).str.strip().astype(int).sum()
    if df_field_accuracy['Total'].astype(str).str.strip().astype(int).sum() > 0
    else 0
)

print(f"Overall micro accuracy (subset): {overall:.4%}")


Per-field accuracy (processed subset):


,Field,Matches,Total,Accuracy
0,Town Name,11,11,1.0
1,Postal Code,6,7,0.8571
2,Country Code (2 characters),10,11,0.9091


Overall micro accuracy (subset): 93.1034%


In [34]:
df_mismatches

,Unstructured Address,Mismatched Field,Original Value,Predicted Value
0,Eschenbrau Braurei Triftstrasse 67 13353 Berli...,Country Code,de,
1,"173/C, NSR Rd, Saibaba Colony, Coimbatore, Tam...",Postal Code,641011,641 011
